

# Importación de datos crudos mediante google drive


In [ ]:
import pandas as pd
dfp=pd.read_csv('/content/drive/MyDrive/projecte futbol/female_players.csv')
dft=pd.read_csv('/content/drive/MyDrive/projecte futbol/female_teams.csv')

In [ ]:
list(dfp.columns)

['player_id',
 'player_url',
 'fifa_version',
 'fifa_update',
 'update_as_of',
 'short_name',
 'long_name',
 'player_positions',
 'overall',
 'potential',
 'value_eur',
 'wage_eur',
 'age',
 'dob',
 'height_cm',
 'weight_kg',
 'club_team_id',
 'club_name',
 'league_id',
 'league_name',
 'league_level',
 'club_position',
 'club_jersey_number',
 'club_loaned_from',
 'club_joined_date',
 'club_contract_valid_until_year',
 'nationality_id',
 'nationality_name',
 'nation_team_id',
 'nation_position',
 'nation_jersey_number',
 'preferred_foot',
 'weak_foot',
 'skill_moves',
 'international_reputation',
 'work_rate',
 'body_type',
 'real_face',
 'release_clause_eur',
 'player_tags',
 'player_traits',
 'pace',
 'shooting',
 'passing',
 'dribbling',
 'defending',
 'physic',
 'attacking_crossing',
 'attacking_finishing',
 'attacking_heading_accuracy',
 'attacking_short_passing',
 'attacking_volleys',
 'skill_dribbling',
 'skill_curve',
 'skill_fk_accuracy',
 'skill_long_passing',
 'skill_ball_co

In [ ]:
list(dft.columns)

['team_id',
 'team_url',
 'fifa_version',
 'fifa_update',
 'update_as_of',
 'team_name',
 'league_id',
 'league_name',
 'league_level',
 'nationality_id',
 'nationality_name',
 'overall',
 'attack',
 'midfield',
 'defence',
 'coach_id',
 'home_stadium',
 'rival_team',
 'international_prestige',
 'domestic_prestige',
 'transfer_budget_eur',
 'club_worth_eur',
 'starting_xi_average_age',
 'whole_team_average_age',
 'captain',
 'short_free_kick',
 'long_free_kick',
 'left_short_free_kick',
 'right_short_free_kick',
 'penalties',
 'left_corner',
 'right_corner',
 'def_style',
 'def_team_width',
 'def_team_depth',
 'def_defence_pressure',
 'def_defence_aggression',
 'def_defence_width',
 'def_defence_defender_line',
 'off_style',
 'off_build_up_play',
 'off_chance_creation',
 'off_team_width',
 'off_players_in_box',
 'off_corners',
 'off_free_kicks',
 'build_up_play_speed',
 'build_up_play_dribbling',
 'build_up_play_passing',
 'build_up_play_positioning',
 'chance_creation_passing',
 'chan

# Creación de tabla de jugadoras

In [ ]:
cols_players = ['player_id',
 'short_name',
 'long_name',
 'player_positions',
 'overall',
 'potential',
 'value_eur',
 'wage_eur',
 'age',
 'dob',
 'height_cm',
 'weight_kg',
 'club_team_id',
 'club_name',
 'league_id',
'league_name',
 'club_contract_valid_until_year',
 'nationality_id',
 'nationality_name',
 'nation_team_id',
 'preferred_foot',
 'skill_moves',
 'body_type',
 'release_clause_eur',
 'pace',
 'shooting',
 'passing',
 'dribbling',
 'defending',
 'physic',
]
players = dfp[cols_players].drop_duplicates(subset=['player_id'])

# Proceso ETL para la tabla de jugadoras


In [ ]:
players.isnull().sum()

,0
player_id,0
short_name,0
long_name,0
player_positions,0
overall,0
potential,0
value_eur,0
wage_eur,0
age,0
dob,0


In [ ]:
players.dropna(subset=['value_eur', 'wage_eur', 'club_team_id', 'release_clause_eur','pace' ], inplace=True)

In [ ]:
players.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1636 entries, 0 to 2652
Data columns (total 30 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   player_id                       1636 non-null   int64  
 1   short_name                      1636 non-null   object 
 2   long_name                       1636 non-null   object 
 3   player_positions                1636 non-null   object 
 4   overall                         1636 non-null   int64  
 5   potential                       1636 non-null   int64  
 6   value_eur                       1636 non-null   float64
 7   wage_eur                        1636 non-null   float64
 8   age                             1636 non-null   int64  
 9   dob                             1636 non-null   object 
 10  height_cm                       1636 non-null   int64  
 11  weight_kg                       1636 non-null   int64  
 12  club_team_id                    1636 no

In [ ]:
import numpy as np

players['Selección'] = np.where(players['nation_team_id'].isnull(),
                                'NO CONVOCADA',
                                'CONVOCADA')

players.drop(columns=['nation_team_id'], inplace=True)


In [ ]:
# Creación de tabla de hechos y dimensiones apartir de la tabla de jugadoras

dim_players = players[['player_id', 'short_name', 'long_name', 'dob', 'height_cm', 'weight_kg', 'preferred_foot', 'body_type']].drop_duplicates()

dim_teams = players[['club_team_id', 'club_name', 'league_id', 'league_name']].drop_duplicates()

dim_nations = players[['player_id', 'nationality_id', 'nationality_name', 'Selección']].drop_duplicates()

fact_stats = players[['player_id', 'club_team_id', 'overall', 'potential', 'value_eur', 'wage_eur',
                      'age', 'skill_moves', 'pace', 'shooting', 'passing',
                      'dribbling', 'defending', 'physic', 'release_clause_eur', 'player_positions' ]]

# Exportación de las tablas a google drive


In [ ]:
carpeta = '/content/drive/MyDrive/projecte futbol/'

players.to_csv(carpeta + 'Players_Completo.csv', index=False)
dim_players.to_csv(carpeta + 'Dim_Players.csv', index=False)
dim_teams.to_csv(carpeta + 'Dim_Teams.csv', index=False)
dim_nations.to_csv(carpeta + 'Dim_Nations.csv', index=False)
fact_stats.to_csv(carpeta + 'Fact_Stats.csv', index=False)